# SQL Analysis - Pink Slip Data

In [70]:
import sqlite3
import pandas as pd

df = pd.read_excel('pink_slips_2021_2023.xlsx')

conn = sqlite3.connect(':memory:')

# build slips table - sum item prices to get total_amount per slip
slip_totals = df.groupby('slip_number')['price'].sum().reset_index().rename(columns={'price': 'total_amount'})
slip_info = df[['slip_number', 'first_initial', 'last_name', 'phone',
                'date_received', 'due_date', 'due_time']].drop_duplicates(subset='slip_number')
slips = slip_info.merge(slip_totals, on='slip_number')

# format dates as MM/DD/YYYY strings for SQL substr queries
for col in ['date_received', 'due_date']:
    slips[col] = pd.to_datetime(slips[col]).dt.strftime('%m/%d/%Y')

slips.to_sql('slips', conn, index=False)

items = df[['slip_number', 'item_type', 'work_description', 'price']]
items.to_sql('items', conn, index=False)

print('slips')
display(pd.read_sql_query('SELECT * FROM slips LIMIT 5', conn))

print('items')
display(pd.read_sql_query('SELECT * FROM items LIMIT 5', conn))

slips


,slip_number,first_initial,last_name,phone,date_received,due_date,due_time,total_amount
0,100001,D,Anderson,9805555719,01/01/2021,01/15/2021,12:00 PM,20
1,100002,O,Padilla,7045554683,01/01/2021,01/15/2021,3:00 PM,22
2,100003,Z,Wagner,7045554659,01/01/2021,01/15/2021,11:00 AM,9
3,100004,H,Jensen,9845554675,01/01/2021,01/15/2021,3:00 PM,9
4,100005,S,Stephens,9805556008,01/01/2021,01/15/2021,4:00 PM,25


items


,slip_number,item_type,work_description,price
0,100001,Pants,Take in waist,5
1,100001,Shirt,Resize,15
2,100002,Jeans,Resize,15
3,100002,Coat,Repair,7
4,100003,Pants,Hem,4


## Revenue by Month

In [71]:
pd.read_sql_query('''
    SELECT
        substr(date_received, 7, 4) || '-' || substr(date_received, 1, 2) AS month,
        COUNT(*) AS total_slips,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_slip_value
    FROM slips
    GROUP BY month
    ORDER BY month
''', conn)

,month,total_slips,revenue,avg_slip_value
0,2021-01,1922,44569.0,23.19
1,2021-02,1899,43307.0,22.81
2,2021-03,2539,55706.0,21.94
3,2021-04,3247,70596.0,21.74
4,2021-05,3377,73667.0,21.81
5,2021-06,3288,69812.0,21.23
6,2021-07,2721,57038.0,20.96
7,2021-08,2357,51414.0,21.81
8,2021-09,2271,48850.0,21.51
9,2021-10,2091,45951.0,21.98


## Top 10 Customers by Total Spend

In [72]:
pd.read_sql_query('''
    SELECT
        first_initial || '. ' || last_name AS customer,
        phone,
        COUNT(*) AS visits,
        ROUND(SUM(total_amount), 2) AS total_spent,
        ROUND(AVG(total_amount), 2) AS avg_per_visit
    FROM slips
    GROUP BY first_initial, last_name, phone
    ORDER BY total_spent DESC
    LIMIT 10
''', conn)

,customer,phone,visits,total_spent,avg_per_visit
0,Z. Smith,7045553689,23,1568.0,68.17
1,Q. Jones,7045555822,25,1549.0,61.96
2,S. George,7045555123,20,1396.0,69.80
3,R. Day,7045556775,19,1376.0,72.42
4,H. Mann,7045556213,16,1309.0,81.81
5,D. Horton,9805552773,20,1280.0,64.00
6,L. Little,7045559380,22,1225.0,55.68
7,X. Ross,9105556499,13,1211.0,93.15
8,C. Nunez,7045559450,17,1191.0,70.06
9,L. Adams,7045557750,19,1185.0,62.37


## Item Type Breakdown

Revenue and volume by item type.

In [73]:
pd.read_sql_query('''
    SELECT
        i.item_type,
        COUNT(*) AS total_items,
        ROUND(AVG(i.price), 2) AS avg_price,
        ROUND(SUM(i.price), 2) AS total_revenue,
        ROUND(SUM(i.price) * 100.0 / (SELECT SUM(price) FROM items), 1) AS pct_of_revenue
    FROM items i
    INNER JOIN slips s ON i.slip_number = s.slip_number
    GROUP BY i.item_type
    ORDER BY total_revenue DESC
''', conn)

,item_type,total_items,avg_price,total_revenue,pct_of_revenue
0,Dress,39532,9.67,382348.0,16.4
1,Pants,42925,8.18,351027.0,15.0
2,Jacket,40627,8.55,347160.0,14.9
3,Jeans,38747,8.15,315650.0,13.5
4,Coat,30719,10.01,307400.0,13.2
5,Shirt,33736,6.85,231209.0,9.9
6,Skirt,22493,8.59,193169.0,8.3
7,Shorts,19897,6.86,136557.0,5.8
8,Other,8084,8.82,71299.0,3.1


## Most Common Alterations by Item Type

In [74]:
pd.read_sql_query('''
    SELECT
        item_type,
        work_description,
        COUNT(*) AS times_performed,
        ROUND(AVG(price), 2) AS avg_price
    FROM items
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY item_type, work_description
    HAVING COUNT(*) > 10
    ORDER BY item_type, times_performed DESC
''', conn)

,item_type,work_description,times_performed,avg_price
0,Coat,Hem,7588,5.08
1,Coat,Shorten sleeves,6036,6.16
2,Coat,Take in waist,5616,8.18
3,Coat,Repair,4491,9.23
4,Coat,Replace zipper,2592,16.48
...,...,...,...,...
119,Skirt,Minor fix,4042,3.38
120,Skirt,Repair,2254,9.23
121,Skirt,Resize,1859,20.66
122,Skirt,Add lining,1324,25.82


## Customer Retention Analysis

Segmenting customers by visit frequency to see which groups drive the most revenue.

In [75]:
pd.read_sql_query('''
    WITH customer_visits AS (
        SELECT
            phone,
            first_initial || '. ' || last_name AS customer,
            COUNT(*) AS visit_count,
            ROUND(SUM(total_amount), 2) AS lifetime_value
        FROM slips
        GROUP BY phone
    )
    SELECT
        CASE
            WHEN visit_count = 1 THEN 'One-time'
            WHEN visit_count BETWEEN 2 AND 4 THEN 'Repeat (2-4)'
            WHEN visit_count BETWEEN 5 AND 9 THEN 'Regular (5-9)'
            ELSE 'VIP (10+)'
        END AS customer_segment,
        COUNT(*) AS num_customers,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_visits), 1) AS pct_of_customers,
        ROUND(SUM(lifetime_value), 2) AS total_revenue,
        ROUND(SUM(lifetime_value) * 100.0 / (SELECT SUM(lifetime_value) FROM customer_visits), 1) AS pct_of_revenue,
        ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
    FROM customer_visits
    GROUP BY customer_segment
    ORDER BY num_customers DESC
''', conn)

,customer_segment,num_customers,pct_of_customers,total_revenue,pct_of_revenue,avg_lifetime_value
0,Repeat (2-4),9023,42.1,632277.0,27.1,70.07
1,One-time,5717,26.7,143269.0,6.1,25.06
2,Regular (5-9),4346,20.3,757071.0,32.4,174.20
3,VIP (10+),2322,10.8,803202.0,34.4,345.91


## Quarterly Revenue Analysis

Comparing revenue across Q1-Q4 to check for seasonal patterns.

In [76]:
pd.read_sql_query('''
    WITH quarterly_data AS (
        SELECT
            substr(date_received, 7, 4) AS year,
            CASE
                WHEN substr(date_received, 1, 2) IN ('01', '02', '03') THEN 'Q1 (Jan-Mar)'
                WHEN substr(date_received, 1, 2) IN ('04', '05', '06') THEN 'Q2 (Apr-Jun)'
                WHEN substr(date_received, 1, 2) IN ('07', '08', '09') THEN 'Q3 (Jul-Sep)'
                ELSE 'Q4 (Oct-Dec)'
            END AS quarter,
            total_amount
        FROM slips
    )
    SELECT
        quarter,
        COUNT(*) AS total_orders,
        ROUND(SUM(total_amount), 2) AS revenue,
        ROUND(AVG(total_amount), 2) AS avg_order_value,
        ROUND(SUM(total_amount) * 100.0 / (SELECT SUM(total_amount) FROM slips), 1) AS pct_of_annual_revenue
    FROM quarterly_data
    GROUP BY quarter
    ORDER BY quarter
''', conn)

,quarter,total_orders,revenue,avg_order_value,pct_of_annual_revenue
0,Q1 (Jan-Mar),18963,504322.0,26.60,21.6
1,Q2 (Apr-Jun),29921,765694.0,25.59,32.8
2,Q3 (Jul-Sep),22224,569025.0,25.60,24.4
3,Q4 (Oct-Dec),18884,496778.0,26.31,21.3


## Revenue by Alteration Type

In [77]:
pd.read_sql_query('''
    SELECT
        work_description AS alteration_type,
        COUNT(*) AS times_performed,
        ROUND(SUM(price), 2) AS total_revenue,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(SUM(price) * 100.0 / (SELECT SUM(price) FROM items), 1) AS pct_of_revenue
    FROM items
    WHERE work_description IS NOT NULL AND work_description != ''
    GROUP BY work_description
    ORDER BY total_revenue DESC
    LIMIT 5
''', conn)

,alteration_type,times_performed,total_revenue,avg_price,pct_of_revenue
0,Take in waist,51451,421012.0,8.18,18.0
1,Hem,79791,405513.0,5.08,17.4
2,Resize,19188,394042.0,20.54,16.9
3,Repair,29834,276088.0,9.25,11.8
4,Add lining,6835,176220.0,25.78,7.5
